# GoFlow — Intégration réelle d'un petit LLM instruct

Réutiliser **exactement les cinq états sauvegardés dans 08**. XGBoost reste le
forecaster ; le LLM interprète les valeurs et propose une recommandation.
Les règles et la validation demeurent les garde-fous. Aucun notebook 01–08 modifié.


## 1. Environnement isolé et réutilisation des résultats

Ce notebook utilise le noyau `.venv-llm`, créé avec les packages système en lecture
seule. Aucune installation dans l'environnement principal. Le module réutilise
les fonctions du notebook 08 via `scripts/decision_agent.py`, les seuils train
déjà enregistrés et les cinq états JSON. Il contrôle aussi les empreintes des sources.


In [1]:
# Charger le module sans réexécuter les notebooks ni refaire du forecasting.
from pathlib import Path
import sys, json, copy
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'notebooks/08_llm_decision_agent.ipynb').exists())
sys.path.insert(0, str(root / 'scripts'))
import local_llm_agent as agent
display(agent.setup_saved_inputs())
display(agent.SCENARIOS[['scenario', 'location', 'timestep']])


C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'model_id': 'Qwen/Qwen2.5-0.5B-Instruct',
 'device': 'cpu',
 'python_version': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]',
 'torch_version': '2.11.0+cpu',
 'cuda_available': False,
 'usable_gpu_count': 0,
 'ollama_executable': None,
 'llama_cpp_model_files': ['models\\qwen2.5-0.5b-instruct-gguf\\qwen2.5-0.5b-instruct-q4_k_m.gguf'],
 'ram_total_bytes': 16487878656,
 'ram_available_before_load_bytes': 4852584448,
 'packages': {'transformers': True,
  'torch': True,
  'llama_cpp': False,
  'ollama': False,
  'accelerate': False,
  'bitsandbytes': False},
 'fine_tuned_on_pems08': False,
 'scenario_count': 5,
 'source_inputs_sha256': '99cfdf3016c165ece861bd7296dd9036a0622eb963d684c63fc44277aa1ccdb6',
 'soft_timeout_s': 60,
 'max_new_tokens': 300,
 'generation': 'greedy, do_sample=False',
 'load_error': None,
 'transformers_version': '5.5.4'}

,scenario,location,timestep
0,trafic_fluide,33,15227
1,occupation_elevee,23,17200
2,forte_hausse_predite,134,16252
3,amelioration_prevue,17,15331
4,voisins_contrastes,34,17530


## 2. Modèle CPU léger et quantification

[Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) annonce
environ 0,49 milliard de paramètres. Les poids officiels sont téléchargés avant
l'exécution, avec une révision figée et des empreintes enregistrées. Le notebook
charge exclusivement les fichiers locaux. La quantification GGUF Q4_K_M et le
backend llama.cpp CPU sont utilisés. L'essai initial INT8 est archivé comme diagnostic.
Les versions, le nombre exact de paramètres et la RSS sont affichés ci-dessous.


In [2]:
# Charger le vrai modèle ; conserver une erreur explicite et le fallback si nécessaire.
display(agent.load_model())
print(agent.SYSTEM_PROMPT)


{'model_id': 'Qwen/Qwen2.5-0.5B-Instruct',
 'device': 'cpu',
 'python_version': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]',
 'torch_version': '2.11.0+cpu',
 'cuda_available': False,
 'usable_gpu_count': 0,
 'ollama_executable': None,
 'llama_cpp_model_files': ['models\\qwen2.5-0.5b-instruct-gguf\\qwen2.5-0.5b-instruct-q4_k_m.gguf'],
 'ram_total_bytes': 16487878656,
 'ram_available_before_load_bytes': 4852584448,
 'packages': {'transformers': True,
  'torch': True,
  'llama_cpp': False,
  'ollama': False,
  'accelerate': False,
  'bitsandbytes': False},
 'fine_tuned_on_pems08': False,
 'scenario_count': 5,
 'source_inputs_sha256': '99cfdf3016c165ece861bd7296dd9036a0622eb963d684c63fc44277aa1ccdb6',
 'soft_timeout_s': 60,
 'max_new_tokens': 300,
 'generation': 'greedy, do_sample=False',
 'load_error': None,
 'transformers_version': '5.5.4',
 'backend': 'llama.cpp CPU local, API limitée à 127.0.0.1',
 'quantization': 'GGUF Q4_K_M officiel Qwen (quant

Tu es l'agent Traffic Prediction & Adaptive Signal Control de GoFlow.
Tu reçois uniquement des données réelles déjà calculées. XGBoost prédit l'occupation
à +15 minutes : tu ne fais jamais de forecasting et ne modifies aucune prévision.
Analyse le trafic et le risque, choisis UNE action autorisée et justifie-la brièvement.
N'invente aucune mesure ni voisin. incident_detected=null et emergency_vehicle_present=null
signifient INCONNUS, pas absents. Ne détecte ni accident ni urgence. Pas de secondes de feu.
Pas de modification de phase car les plans de feux sont inconnus.
Réponds uniquement en JSON, exactement les clés traffic_state, risk_level,
recommended_action, reason, confidence. confidence est un float entre 0.0 et 1.0,
ton auto-évaluation non calibrée. Respecte les enums et garde-fous fournis.
Dans reason, cite l'occupation actuelle et la prévision avec leurs nombres fournis
à six décimales. Termine exactement par : Incidents inconnus; urgences inconnues;
plan de feux inconnu. Évit

## 3. Appel réel et contrat de sortie

`agent.llm_decision(state)` sérialise l'état, applique le chat template, appelle
le serveur llama.cpp local avec schéma JSON, extrait un seul JSON et
retourne un dict. Le texte brut est conservé ; aucune correction silencieuse.

`agent.final_decision(state)` tente le LLM, valide les cinq clés, catégories,
confidence, justification et cohérence ; sinon elle retourne les règles avec
`source=rule_based_fallback` et un motif explicite. Les méthodes sont visibles
ci-dessous pour rendre la logique reviewable sans la dupliquer.


In [3]:
# Afficher les fonctions effectives utilisées dans cette exécution.
import inspect
for function in [agent.llm_decision, agent.extract_json, agent.validate_decision,
                 agent.qualitative_checks, agent.final_decision]:
    print(inspect.getsource(function))


def llm_decision(state):
    global LAST_CALL
    LAST_CALL = {"raw_text": None, "parsed_output": None, "error": None,
                 "inference_time_s": None, "input_tokens": None, "output_tokens": None}
    start = perf_counter()
    try:
        if MODEL is None:
            raise RuntimeError("Modèle local indisponible : " + str(MODEL_INFO.get("load_error")))
        baseline = rules.rule_based_decision(state)
        payload = {"state": state, "indicators": rules.indicators(state),
                   "traffic_state_allowed": sorted(rules.TRAFFIC_STATES),
                   "risk_level_allowed": sorted(rules.RISK_LEVELS),
                   "allowed_actions_here": ["keep_current_signal_plan", "monitor",
                                            "coordinate_with_neighbor", "manual_review"],
                   "minimum_risk": baseline["risk_level"],
                   "current_occupancy_six_decimals": f"{state['current_occupancy']:.6f}",
                   "predicted_occupancy_si

## 4. Les cinq mêmes scénarios — un appel réel par état

Comparer les sorties brutes du LLM, les règles et les décisions finales. Aucun
nouveau scénario ni cible réelle à t+3 n'est utilisé. Le temps mesuré inclut
tokenisation, génération et parsing, mais pas le chargement du modèle.


In [4]:
# Effectuer exactement cinq appels réels et conserver leurs erreurs éventuelles.
records, comparison, validation, times = agent.run_scenarios()
for record in records:
    print('\nSCÉNARIO :', record['scenario'])
    print(json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False))
display(comparison)
display(validation)
display(times)


trafic_fluide source= rule_based_fallback temps= 6.070576300029643


occupation_elevee source= rule_based_fallback temps= 5.108968500047922


forte_hausse_predite source= rule_based_fallback temps= 4.58452789997682


amelioration_prevue source= rule_based_fallback temps= 5.72223319998011


voisins_contrastes source= rule_based_fallback temps= 4.811959499958903



SCÉNARIO : trafic_fluide
{
  "scenario": "trafic_fluide",
  "state": {
    "sensor_id": 33,
    "timestep": 15227,
    "current_flow": 0.0,
    "current_speed": 65.0,
    "current_occupancy": 0.0,
    "predicted_occupancy_t15": 0.0027397831,
    "occupancy_change": 0.0027397831,
    "recent_occupancy_trend": 0.0001,
    "recent_speed_trend": 0.019999999999998862,
    "neighbors": [
      {
        "sensor_id": 32,
        "timestep": 15227,
        "current_speed": 61.8,
        "current_occupancy": 0.0381,
        "predicted_occupancy_t15": 0.039587665
      },
      {
        "sensor_id": 156,
        "timestep": 15227,
        "current_speed": 64.5,
        "current_occupancy": 0.0144,
        "predicted_occupancy_t15": 0.013298589
      }
    ],
    "incident_detected": null,
    "emergency_vehicle_present": null
  },
  "rule_based_output": {
    "traffic_state": "free_flow",
    "risk_level": "low",
    "recommended_action": "keep_current_signal_plan",
    "reason": "Occupation a

,Scenario,Rule-based action,LLM action,Same action?,LLM JSON valid?,Hallucination detected?,Reason grounded?,Inference time,Final source
0,trafic_fluide,keep_current_signal_plan,keep_current_signal_plan,True,True,False,False,6.070576,rule_based_fallback
1,occupation_elevee,manual_review,keep_current_signal_plan,False,True,False,False,5.108969,rule_based_fallback
2,forte_hausse_predite,coordinate_with_neighbor,keep_current_signal_plan,False,True,False,False,4.584528,rule_based_fallback
3,amelioration_prevue,coordinate_with_neighbor,keep_current_signal_plan,False,True,False,False,5.722233,rule_based_fallback
4,voisins_contrastes,monitor,keep_current_signal_plan,False,True,False,False,4.811959,rule_based_fallback


,scenario,json_valid,authorized_action,no_unsupported_claim_detected,reason_grounded,risk_action_coherent,confidence_valid,missing_information_respected,hallucination_detected,hallucination_check_applicable,final_source,fallback_reason
0,trafic_fluide,True,True,True,False,False,True,False,False,True,rule_based_fallback,"Critères refusés : reason_grounded, risk_actio..."
1,occupation_elevee,True,True,True,False,False,True,False,False,True,rule_based_fallback,"Critères refusés : reason_grounded, risk_actio..."
2,forte_hausse_predite,True,True,True,False,False,True,False,False,True,rule_based_fallback,"Critères refusés : reason_grounded, risk_actio..."
3,amelioration_prevue,True,True,True,False,False,True,False,False,True,rule_based_fallback,"Critères refusés : reason_grounded, risk_actio..."
4,voisins_contrastes,True,True,True,False,False,True,False,False,True,rule_based_fallback,"Critères refusés : reason_grounded, risk_actio..."


,scenario,inference_time_s,input_tokens,output_tokens,error
0,trafic_fluide,6.070576,775,65,None
1,occupation_elevee,5.108969,769,81,None
2,forte_hausse_predite,4.584528,767,77,None
3,amelioration_prevue,5.722233,952,77,None
4,voisins_contrastes,4.811959,768,81,None


## 5. Tests de robustesse avec fixtures simulées

Les tests ci-dessous utilisent un état réel mais remplacent l'appel LLM par des
sorties erronées ou exceptions contrôlées. Ils ne constituent pas des sorties du
modèle et ne sont pas inclus dans la comparaison des cinq inférences réelles.


In [5]:
# Vérifier le fallback pour chaque défaillance demandée, sans appel supplémentaire au modèle.
state = agent.INPUTS[0]['state']
baseline = agent.rules.rule_based_decision(state)
original = agent.llm_decision
fixtures = [
    ('json_invalide', '{'),
    ('cle_manquante', {k: v for k, v in baseline.items() if k != 'risk_level'}),
    ('action_interdite', {**baseline, 'recommended_action': 'invented_action'}),
    ('type_invalide', {**baseline, 'risk_level': []}),
    ('confidence_hors_limites', {**baseline, 'confidence': 1.5}),
    ('faux_incident', {**baseline, 'reason': 'Accident détecté.'})]
try:
    for name, fixture in fixtures:
        agent.llm_decision = lambda input_state, answer=fixture: answer
        result = agent.final_decision(state)
        assert result['source'] == 'rule_based_fallback', name
        assert result['decision'] == baseline, name
        assert result['fallback_reason'], name
    for error in [RuntimeError('Modèle indisponible simulé'),
                  TimeoutError('Timeout simulé'), OSError('Erreur d’inférence simulée')]:
        def fail(input_state, exception=error):
            raise exception
        agent.llm_decision = fail
        result = agent.final_decision(state)
        assert result['source'] == 'rule_based_fallback'
        assert type(error).__name__ in result['fallback_reason']
finally:
    agent.llm_decision = original
assert not agent.validate_decision({**baseline, 'confidence': True})
assert not agent.validate_decision({**baseline, 'confidence': float('nan')})
assert agent.extract_json('Texte avant ```json ' + json.dumps(baseline) + ' ```') == baseline
try:
    agent.extract_json('{"confidence": 0.5, "confidence": 0.8}')
except ValueError:
    pass
else:
    raise AssertionError('Clés dupliquées non refusées')
print('Tests JSON, contrat, faux incident, indisponibilité et timeout : OK.')


Tests JSON, contrat, faux incident, indisponibilité et timeout : OK.


## 6. Sauvegarde et résumé réel

Sauvegarder séparément sous `results/llm_agent/`. Ni les résultats de 08 ni le
benchmark ne sont modifiés. JSON valide, grounding et nombre de fallbacks sont
des contrôles qualitatifs ; ils ne constituent pas une accuracy.


In [6]:
methodology = "# GoFlow — Intégration d'un LLM instruct local\n\nXGBoost fournit les prévisions d'occupation à +15 minutes. Le LLM ne fait pas de\nforecasting : il reçoit les cinq états déjà sauvegardés par le notebook 08, avec\nmesures, tendances passées et voisins, puis propose une catégorie de risque, une\naction et une justification. Les règles constituent le fallback. Aucun scénario\nn'est recréé, aucune cible future n'est passée au modèle, aucun modèle trafic\nn'est réentraîné. Les fichiers de results/decision_agent/ sont lus, pas modifiés.\n\n## Modèle et environnement\n\nModèle : Qwen/Qwen2.5-0.5B-Instruct, environ 0,49 milliard de paramètres, issu du\n[dépôt du fournisseur](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).\nLa révision téléchargée et les SHA256 de chaque fichier sont enregistrés sous\nmodels/qwen2.5-0.5b-instruct/download_provenance.json. Les informations effectives\nde chargement, nombre exact de paramètres, mémoire RSS et versions sont enregistrées\ndans llm_model_info.json. Ce modèle est préentraîné/instruct, sans adaptation à PEMS08.\n\nL'environnement .venv-llm utilise --system-site-packages pour lire les dépendances\nexistantes sans les modifier. Aucune installation de package n'est nécessaire.\nLe CPU est utilisé, avec quatre threads ; CUDA n'est pas disponible sur cette\ninstallation PyTorch. Le backend retenu est le binaire CPU Windows officiel\n[llama.cpp b11596](https://github.com/ggml-org/llama.cpp/releases/tag/b11596), avec le\n[GGUF Qwen officiel Q4_K_M](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF).\nProvenance et SHA256 du GGUF et du binaire sont enregistrés sous\nmodels/qwen2.5-0.5b-instruct-gguf/download_provenance.json. Le serveur est lié à\n127.0.0.1 sur un port libre et arrêté en fin de notebook. Sa RSS est mesurée\nséparément de celle du noyau Python. Le chargement exige 1 Go libre.\n\nLe premier essai Transformers/INT8 a généré cinq réponses, avec zéro JSON valide,\ncinq fallbacks et trois délais dépassés. Tous ses résultats, textes, erreurs et\ntemps sont conservés dans diagnostics/transformers_int8_first_attempt. Le changement\nde backend répond à cet échec technique, sans consulter les cibles futures et\nsans modifier les scénarios. Aucun échec initial n'est effacé.\n\n## Génération et validation\n\nChat template embarqué dans le GGUF officiel, prompt système français, température\nzéro, maximum 300 nouveaux tokens et génération contrainte par schéma JSON dans\nllama.cpp. Le schéma impose les champs et enums, pas les décisions ni une raison\npréécrite. Le texte utile est récupéré dans message.content.\nLe prompt et les paramètres sont fixés avant de parcourir les cinq scénarios ;\naucun réglage par comparaison à la cible future n'est réalisé. Le modèle n'est\npas entraîné pendant cette étape. Un timeout HTTP de 60 secondes et un contrôle\nde durée après réponse déclenchent un fallback ; une opération serveur en cours\nn'est pas garantie d'être interrompue instantanément. Le log serveur est conservé.\nLe texte brut, le JSON extrait, les erreurs, motifs de fallback,\ntokens et durées restent disponibles pour chacun des cinq appels réels.\n\nL'extraction accepte du bruit autour d'un unique objet JSON mais ne répare aucune\nvaleur, clé ou ponctuation et refuse les clés dupliquées, NaN et blocs ambigus.\nLe validateur strict du notebook 08 est réutilisé ; confidence doit être un float.\nLes catégories de risque et de trafic doivent être cohérentes. Le niveau de risque\ndes règles est un plancher expérimental, pas une vérité scientifique. Une action\nde coordination nécessite au moins un voisin à occupation élevée selon les mêmes\nseuils train de 08. Les phases de feux restent interdites faute de contexte de commande.\n\nLe LLM peut produire une justification libre, mais elle doit citer les deux valeurs\nd'occupation fournies à six décimales et la phrase : « Incidents inconnus; urgences\ninconnues; plan de feux inconnu. » Les autres affirmations incident/urgence/voisin\nsont refusées de manière lexicale conservatrice ; les nombres de la justification\ndoivent correspondre aux occupations fournies ou à l'horizon 15. Ces tests peuvent\nrejeter des formulations pourtant légitimes et ne garantissent pas l'absence de\ntoute hallucination sémantique. Le contrôle d'hallucination porte seulement sur\nles raisons JSON analysables ; les réponses non parsées exigent une revue manuelle.\n« Reason grounded » signifie conformité à ces\ncontrôles, pas une annotation humaine démontrant une causalité correcte.\n\nLa comparaison rules/LLM est qualitative. L'accord d'action n'est pas une accuracy,\net aucun résultat ne prouve que le LLM est meilleur. Sa confidence est une\nauto-évaluation non calibrée, pas une probabilité statistique. Les cinq scénarios\nsont les cas illustratifs extrêmes déjà retenus dans 08, pas un échantillon représentatif.\nLe mapping location=index NPZ reste hypothétique pour les 25 capteurs aux séries\nidentiques documentés dans 08 ; les recommandations de voisinage héritent de cette limite.\n\n## Robustesse et architecture\n\nLes tests de JSON invalide, clé manquante, action interdite, confidence >1, faux\nincident, modèle indisponible et timeout utilisent des fixtures explicitement\nsimulées. Ils ne sont pas enregistrés comme sorties du modèle réel.\n\n```text\nPEMS08 → Feature Engineering → XGBoost → Prediction +15 min\n  → Traffic State Builder (état courant, tendances passées, voisins directs)\n  → LLM Decision Agent → JSON et garde-fous → Fallback si nécessaire\n  → Recommandation structurée → Orchestrator futur\n```\n\nAucune commande n'est envoyée à des feux ou à un système externe. Les notebooks\n01–08, le benchmark et les données sources restent inchangés. L'intégration technique\nest réussie si le modèle se charge et génère du texte pour les cinq appels ; le\nnombre de décisions acceptées est rapporté séparément, sans masquer les fallbacks.\n"
summary = agent.save_results(records, comparison, validation, times, methodology)


{
  "Modèle LLM utilisé": "Qwen/Qwen2.5-0.5B-Instruct",
  "Device": "cpu",
  "Nombre de scénarios": 5,
  "JSON valides": 5,
  "Fallbacks": 5,
  "Hallucinations détectées sur JSON analysables": 0,
  "Réponses non analysables": 0,
  "Temps moyen d'inférence (s)": 5.25965307999868,
  "Intégration technique": true
}


## 7. Limites et architecture

```text
PEMS08 → Feature Engineering → XGBoost → Prévision +15 min
  → Traffic State Builder
      ├── état courant
      ├── tendances passées
      └── voisins directs
  → LLM Decision Agent → JSON validation → Fallback si nécessaire
  → Recommandation structurée → Orchestrator futur
```

La confidence du LLM est une auto-évaluation **non calibrée**. Les cinq cas ne
permettent pas d'affirmer que le LLM est meilleur. Les contrôles anti-hallucination
sont conservateurs et partiels : ils ne prouvent pas une justesse sémantique totale.
Le mapping reste hypothétique pour les 25 capteurs indiscernables documentés dans 08.
PEMS08 ne fournit pas de plans de feux, d'incidents ni d'urgences : aucune commande
de feux n'est envoyée, aucune durée de phase n'est recommandée.


In [7]:
# Séparer le succès technique de la conformité des décisions.
if summary['Intégration technique']:
    print('LLM techniquement intégré : génération réelle effectuée sur les cinq scénarios.')
else:
    print('Intégration technique incomplète : consulter load_error et les erreurs d’inférence sauvegardées.')
print('Décisions LLM retenues :', int((validation.final_source == 'llm').sum()))
print('Fallbacks documentés :', summary['Fallbacks'])


LLM techniquement intégré : génération réelle effectuée sur les cinq scénarios.
Décisions LLM retenues : 0
Fallbacks documentés : 5


## Diagnostic de l’intégration

Un premier essai GGUF utilisait une enveloppe response_format incomplète : le serveur appliquait un JSON générique. Ses sorties sont conservées dans diagnostics/llama_cpp_schema_first_attempt. La correction utilise response_format.json_schema.schema, conformément au code de llama.cpp b11596. Le schéma borne aussi reason à 220 caractères. Les cinq dernières inférences et leurs décisions rejetées restent enregistrées sans modification.

Les cinq derniers JSON sont valides, mais aucune décision LLM ne passe les garde-fous. Toutes les décisions finales utilisent le fallback ; le succès technique ne démontre pas une qualité décisionnelle suffisante.